# Train the Checker (Stage 2)

Fine-tunes a `yolo26-cls` binary classifier (`ashwagandha` vs.
`not_ashwagandha`) on the crops Stage 1 hands it. Same checkpointing/resume
setup as `train_finder.ipynb` -- Ultralytics saves `last.pt` every epoch
regardless of task, so this is safe to interrupt and re-run.

Needs `scripts/prepare_classifier_data.py` to have already been run (needs
`data/raw/negatives/<source>/` to have images in it first).

**Two numbers below, same as the Finder notebook:** the val-accuracy cell is
what training itself watches (it's literally what `best.pt` gets picked
against, so treat it as informative but optimistic, not neutral), and a
separate test-set cell further down gives the honest, never-touched-during-
training number -- same reasoning `train_finder.ipynb` already uses for
Stage 1's test-set check.

In [1]:
from pathlib import Path

from ultralytics import YOLO

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = REPO_ROOT / "data" / "classify"
MODEL_SIZE = "m"    # only matters for a fresh start -- there's no domain-specific prior
                     # checkpoint for the classifier the way best_m.pt exists for the Finder,
                     # so a fresh run always starts from Ultralytics' generic pretrained weights
STARTING_WEIGHTS = f"yolo26{MODEL_SIZE}-cls.pt"
PROJECT_DIR = REPO_ROOT / "runs" / "checker"
RUN_NAME = "train"

EPOCHS = 60
PATIENCE = 20       # unlike Stage 1's patience value, this one isn't backed by an actual
                     # observed training curve yet -- adjust if you see it stop too early or too late
IMGSZ = 224
BATCH = -1           # -1 = let Ultralytics pick the biggest batch that fits your GPU
AUTO_AUGMENT = "randaugment"   # small dataset -- fight overfitting with strong augmentation,
                                 # same reasoning as the Azadnia et al. 2024 paper the README cites
SAVE_PERIOD = 10    # keep a numbered snapshot (epoch10.pt, epoch20.pt, ...) every N epochs too

In [2]:
last_checkpoint = PROJECT_DIR / RUN_NAME / "weights" / "last.pt"


def start_fresh():
    for split in ("train", "val", "test"):
        split_dir = DATA_DIR / split
        if not split_dir.is_dir() or not any(split_dir.iterdir()):
            raise SystemExit(
                f"{split_dir} is missing or empty. Run scripts/prepare_classifier_data.py first."
            )
    model = YOLO(STARTING_WEIGHTS)
    model.train(
        data=str(DATA_DIR),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        auto_augment=AUTO_AUGMENT,
        save_period=SAVE_PERIOD,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
    )
    return model


if last_checkpoint.exists():
    try:
        print(f"found a checkpoint at {last_checkpoint} -- trying to resume it")
        model = YOLO(str(last_checkpoint))
        # pass the checkpoint PATH here, not just True -- see train_finder.ipynb for why
        model.train(resume=str(last_checkpoint))
    except AssertionError as e:
        print(f"nothing to resume ({e})")
        print("starting a fresh run instead -- it'll land in a new numbered folder")
        model = start_fresh()
else:
    print("no existing checkpoint found, starting fresh")
    model = start_fresh()

save_dir = model.trainer.save_dir
print(f"\nthis run's files are in: {save_dir}")

found a checkpoint at /home/tj_students/ashwagandha-leaf-detection/runs/checker/train/weights/last.pt -- trying to resume it


New https://pypi.org/project/ultralytics/8.4.121 available 😃 Update with 'pip install -U ultralytics'


Ultralytics 8.4.104 🚀 Python-3.10.12 torch-2.13.0+cu130 CUDA:0 (NVIDIA RTX A5000, 24111MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=253, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/tj_students/ashwagandha-leaf-detection/data/classify, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/tj_students/ashwagandha-leaf-detection/runs/checker/train/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-8, nbs=64, nms=False, opset=None

train: /home/tj_students/ashwagandha-leaf-detection/data/classify/train... found 550 images in 2 classes ✅ 


...

               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 5.4it/s 0.2s

                   all      0.991          1


Speed: 0.0ms preprocess, 0.2ms inference, 0.0ms loss, 0.0ms postprocess per image


Results saved to /home/tj_students/ashwagandha-leaf-detection/runs/checker/train-9



this run's files are in: /home/tj_students/ashwagandha-leaf-detection/runs/checker/train-9


### check the result

`model.val()` re-runs validation cleanly and prints top-1 accuracy directly,
rather than trusting whatever scrolled by during training.

In [3]:
# project/name here matter -- without them Ultralytics saves to a default location
# based on wherever this notebook's working directory happens to be (notebooks/, not
# the repo root), scattering results away from everything else about this run
val_metrics = model.val(data=str(DATA_DIR), imgsz=IMGSZ, project=str(save_dir), name="val_check")
print(f"\nval top-1 accuracy: {val_metrics.top1:.3f}")

Ultralytics 8.4.104 🚀 Python-3.10.12 torch-2.13.0+cu130 CUDA:0 (NVIDIA RTX A5000, 24111MiB)


YOLO26m-cls summary (fused): 57 layers, 10,344,194 parameters, 0 gradients, 39.3 GFLOPs


train: /home/tj_students/ashwagandha-leaf-detection/data/classify/train... found 550 images in 2 classes ✅ 


val: /home/tj_students/ashwagandha-leaf-detection/data/classify/val... found 108 images in 2 classes ✅ 


test: /home/tj_students/ashwagandha-leaf-detection/data/classify/test... found 125 images in 2 classes ✅ 


...

               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 7/7 13.4it/s 0.5s

                   all      0.991          1


Speed: 0.1ms preprocess, 4.6ms inference, 0.0ms loss, 0.0ms postprocess per image


Results saved to /home/tj_students/ashwagandha-leaf-detection/runs/checker/train-9/val_check



val top-1 accuracy: 0.991


In [4]:
import matplotlib.pyplot as plt
from PIL import Image


def show(path, title):
    if not path.exists():
        print(f"(missing: {path})")
        return
    plt.figure(figsize=(10, 8))
    plt.imshow(Image.open(path))
    plt.title(title)
    plt.axis("off")
    plt.show()


show(save_dir / "results.png", "training curves (all epochs)")
show(save_dir / "confusion_matrix.png", "confusion matrix (val split)")
show(save_dir / "val_batch0_labels.jpg", "val split -- ground truth")
show(save_dir / "val_batch0_pred.jpg", "val split -- what the model predicted")

<Figure size 1000x800 with 1 Axes>

<Figure size 1000x800 with 1 Axes>

<Figure size 1000x800 with 1 Axes>

<Figure size 1000x800 with 1 Axes>

### test set



In [5]:
test_metrics = model.val(data=str(DATA_DIR), split="test", imgsz=IMGSZ,
                          project=str(save_dir), name="test_check")
print(f"\ntest top-1 accuracy: {test_metrics.top1:.3f}")

test_dir = save_dir / "test_check"
show(test_dir / "confusion_matrix.png", "confusion matrix (test split)")

Ultralytics 8.4.104 🚀 Python-3.10.12 torch-2.13.0+cu130 CUDA:0 (NVIDIA RTX A5000, 24111MiB)


train: /home/tj_students/ashwagandha-leaf-detection/data/classify/train... found 550 images in 2 classes ✅ 


val: /home/tj_students/ashwagandha-leaf-detection/data/classify/val... found 108 images in 2 classes ✅ 


test: /home/tj_students/ashwagandha-leaf-detection/data/classify/test... found 125 images in 2 classes ✅ 


test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 742.4±205.2 MB/s, size: 15.2 KB)


...

                   all      0.968          1


Speed: 0.4ms preprocess, 2.9ms inference, 0.0ms loss, 0.0ms postprocess per image


Results saved to /home/tj_students/ashwagandha-leaf-detection/runs/checker/train-9/test_check



test top-1 accuracy: 0.968


<Figure size 1000x800 with 1 Axes>